In [9]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
CAPTIONS_FILE = DATA_DIR / "captions.txt"
IMAGES_DIR = DATA_DIR / "Images"

with open(CAPTIONS_FILE) as f:
    for _ in range(5):
        print(repr(f.readline()))

'image,caption\n'
'1000268201_693b08cb0e.jpg,A child in a pink dress is climbing up a set of stairs in an entry way .\n'
'1000268201_693b08cb0e.jpg,A girl going into a wooden building .\n'
'1000268201_693b08cb0e.jpg,A little girl climbing into a wooden playhouse .\n'
'1000268201_693b08cb0e.jpg,A little girl climbing the stairs to her playhouse .\n'


In [10]:
df = pd.read_csv(CAPTIONS_FILE)
print(df.shape)
print(df.columns.tolist())
df.head()

(40455, 2)
['image', 'caption']


,image,caption
0,1000268201_693b08cb0e.jpg,A child in a pink dress is climbing up a set o...
1,1000268201_693b08cb0e.jpg,A girl going into a wooden building .
2,1000268201_693b08cb0e.jpg,A little girl climbing into a wooden playhouse .
3,1000268201_693b08cb0e.jpg,A little girl climbing the stairs to her playh...
4,1000268201_693b08cb0e.jpg,A little girl in a pink dress going into a woo...


In [11]:
sample_file = df.iloc[0]["image"]
print(sample_file, (IMAGES_DIR / sample_file).exists())

print(df["image"].nunique(), "unique images")
print(df.groupby("image").size().value_counts())   # captions-per-image distribution

1000268201_693b08cb0e.jpg True
8091 unique images
5    8091
Name: count, dtype: int64


In [18]:
import re 
import random 
from collections import Counter

def tokenize(s):
    return re.findall(r"[a-z]+", s.lower())
PAD , SOS, EOS, UNK = "<pad>", "<sos>", "<eos>", "<unk>"

class Vocab:
    def __init__(self, token_lists, min_freq=5):
        counts = Counter(t for toks in token_lists for t in toks)
        self.itos = [PAD, SOS, EOS, UNK] + sorted(w for w, c in counts.items() if c >= min_freq)
        self.stoi = {w: i for i, w in enumerate(self.itos)}

    def encode(self, toks):
        return [self.stoi[SOS]] + [self.stoi.get(t, self.stoi[UNK]) for t in toks] + [self.stoi[EOS]]

    def decode(self, ids):
        return " ".join(self.itos[i] for i in ids if i not in (0, 1, 2))

images = df["image"].unique().tolist()
random.seed(42)
random.shuffle(images)

n = len(images)
train_imgs = set(images[:int(.8 * n)])
val_imgs   = set(images[int(.8 * n):int(.9 * n)])
test_imgs  = set(images[int(.9 * n):])

train_df = df[df["image"].isin(train_imgs)].reset_index(drop=True)
val_df   = df[df["image"].isin(val_imgs)].reset_index(drop=True)
test_df  = df[df["image"].isin(test_imgs)].reset_index(drop=True)

vocab = Vocab([tokenize(c) for c in train_df["caption"]])

print(f"train: {len(train_imgs)} images, {len(train_df)} captions")
print(f"val:   {len(val_imgs)} images, {len(val_df)} captions")
print(f"test:  {len(test_imgs)} images, {len(test_df)} captions")
print(f"vocab size: {len(vocab.itos)}")

train: 6472 images, 32360 captions
val:   809 images, 4045 captions
test:  810 images, 4050 captions
vocab size: 2652


In [22]:
import pickle 
import json 

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

train_df.to_csv(ARTIFACTS_DIR / "train.csv", index = False)
test_df.to_csv(ARTIFACTS_DIR / "test.csv", index = False)
val_df.to_csv(ARTIFACTS_DIR / "val.csv", index = False)

with open(ARTIFACTS_DIR / "vocab.pkl", "wb") as f:
    pickle.dump(vocab,f)
print("Saved")

Saved


In [23]:
with open(ARTIFACTS_DIR / "vocab.pkl", "rb") as f:
    vocab_check = pickle.load(f)

print(len(vocab_check.itos))
print(vocab_check.encode(["a", "dog", "runs"]))

2652
[1, 4, 653, 1892, 2]
